# 🧠 CerebroSpheroid-2B: Cognitive Brain Manifold Architecture
### Training on Kaggle (NVIDIA T4 x2 or P100 16GB GPU)

**New Cognitive Upgrades:**
1. **Afferent Sensory Input DNN (Cortical Front-End)**: 2-layer SwiGLU ResNet translating discrete symbolic tokens into smooth continuous Riemannian boundary flux.
2. **Stabilized Spheroid Manifold Core (^3$)**: 32 Concentric Laminae with 173 Subcortical Axonal Shortcut Tracts and unit-gain contractive dynamics, eliminating exploding gradients.
3. **Efferent Motor Output DNN (Dual-Tap Back-End)**: Combines superficial cortical speech resonance with deep thalamic working memory attractors.
4. **Multi-Step Reasoning (GSM8K)**: Trains active thinking: Problem Ingestion $ightarrow$ Internal Subconscious Deliberation $ightarrow$ Zero-Feedback Answer Emission.

---
### ⚡ Kaggle Instructions:
1. In the right-hand **Notebook Settings** panel:
   - **Accelerator**: Select **GPU T4 x2** (recommended, 32GB total) or **GPU P100** (16GB VRAM).
   - **Internet**: Toggle **Internet ON** (required to stream GSM8K reasoning dataset).
2. Click **Run All** ( or ).

## Step 1: Verify Kaggle GPU Hardware

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"GPU Count       : {num_gpus}")
    for i in range(num_gpus):
        vram_gb = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({vram_gb:.2f} GB VRAM)")

## Step 2: Install Required Libraries

In [ ]:
!pip install -q transformers datasets accelerate

## Step 3: Define Upgraded Cognitive Brain Architecture

In [ ]:
import os, math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple, Dict, Any

class SensoryInputDNN(nn.Module):
    """Afferent Sensory Front-End: 2-layer SwiGLU ResNet with RMSNorm."""
    def __init__(self, vocab_size: int = 32000, hidden_dim: int = 2048, surface_dim: int = 2048):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.fc1_gate = nn.Linear(hidden_dim, hidden_dim * 2, bias=False)
        self.fc1_val = nn.Linear(hidden_dim, hidden_dim * 2, bias=False)
        self.norm1 = nn.RMSNorm(hidden_dim * 2)
        self.fc2 = nn.Linear(hidden_dim * 2, surface_dim, bias=False)
        self.norm_out = nn.RMSNorm(surface_dim)
        self.residual_proj = nn.Linear(hidden_dim, surface_dim, bias=False) if hidden_dim != surface_dim else nn.Identity()

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        x = self.embedding(token_ids)
        gate = F.silu(self.fc1_gate(x))
        val = self.fc1_val(x)
        h = self.norm1(gate * val)
        flux = self.fc2(h) + self.residual_proj(x)
        return self.norm_out(flux)


class MotorOutputDNN(nn.Module):
    """Efferent Motor Back-End: Dual-Tap Decoder reading Cortex & Thalamic Core."""
    def __init__(self, surface_dim: int = 2048, core_dim: int = 2048, vocab_size: int = 32000):
        super().__init__()
        fused_dim = surface_dim + core_dim
        hidden_dim = surface_dim * 2
        self.fused_norm = nn.RMSNorm(fused_dim)
        self.fc_gate = nn.Linear(fused_dim, hidden_dim, bias=False)
        self.fc_val = nn.Linear(fused_dim, hidden_dim, bias=False)
        self.norm_mid = nn.RMSNorm(hidden_dim)
        self.head = nn.Linear(hidden_dim, vocab_size, bias=False)
        self.temperature_scale = nn.Parameter(torch.tensor(1.0))

    def forward(self, surface_state: torch.Tensor, core_state: Optional[torch.Tensor] = None) -> torch.Tensor:
        if core_state is None:
            core_state = surface_state
        fused = self.fused_norm(torch.cat([surface_state, core_state], dim=-1)).to(self.fc_gate.weight.dtype)
        gate = F.silu(self.fc_gate(fused))
        val = self.fc_val(fused)
        h = self.norm_mid(gate * val)
        logits = self.head(h) / torch.clamp(self.temperature_scale, min=0.1, max=5.0)
        return logits


class VolumetricBrainCore(nn.Module):
    """Stabilized 3D Spherical Ball Core (B^3) with Contractive Unit-Gain Dynamics."""
    def __init__(self, num_shells: int = 32, shell_dim: int = 2048, radial_dim: int = 1024, num_shortcut_tracts: int = 173):
        super().__init__()
        self.num_shells = num_shells
        self.shell_dim = shell_dim
        self.radial_dim = radial_dim
        self.num_shortcut_tracts = num_shortcut_tracts
        src_indices = torch.arange(num_shortcut_tracts) % num_shells
        dst_indices = (num_shells - 1 - (torch.arange(num_shortcut_tracts) * 3) % num_shells)
        self.register_buffer("src_indices", src_indices)
        self.register_buffer("dst_indices", dst_indices)
        self.intra_shell = nn.Parameter(torch.empty(num_shells, shell_dim, shell_dim).normal_(mean=0.0, std=0.015))
        self.radial_proj_in = nn.Linear(shell_dim, radial_dim, bias=False)
        self.reentrant_field = nn.Parameter(torch.empty(num_shells, num_shells, radial_dim, radial_dim).normal_(mean=0.0, std=0.008))
        self.radial_proj_out = nn.Linear(radial_dim, shell_dim, bias=False)
        self.shortcut_tracts = nn.Parameter(torch.empty(num_shortcut_tracts, shell_dim, shell_dim).normal_(mean=0.0, std=0.01))
        self.inertia_gate = nn.Parameter(torch.tensor(0.80))
        self.norm = nn.RMSNorm(shell_dim)

    def init_state(self, batch_size: int, device: torch.device) -> torch.Tensor:
        return torch.zeros(batch_size, self.num_shells, self.shell_dim, device=device)

    def inject_sensory_flux(self, state: torch.Tensor, boundary_flux: torch.Tensor) -> torch.Tensor:
        new_state = state.clone()
        new_state[:, -1, :] = new_state[:, -1, :] + boundary_flux.to(state.dtype)
        return self.norm(new_state).to(state.dtype)

    def deliberate_step(self, state: torch.Tensor) -> torch.Tensor:
        curr = state.to(self.intra_shell.dtype)
        intra_flow = torch.einsum("bkd,kdo->bko", curr, self.intra_shell)
        radial_compact = self.radial_proj_in(curr)
        radial_flow = self.radial_proj_out(torch.einsum("bki,kjio->bjo", radial_compact, self.reentrant_field))
        src_state = curr[:, self.src_indices, :]
        tract_flow = torch.einsum("btd,tdo->bto", src_state.to(self.shortcut_tracts.dtype), self.shortcut_tracts)
        shortcut_flow = torch.zeros(curr.shape, dtype=tract_flow.dtype, device=curr.device)
        shortcut_flow.index_add_(1, self.dst_indices, tract_flow)
        total_velocity = intra_flow + radial_flow + 0.3 * shortcut_flow
        gamma = torch.sigmoid(self.inertia_gate)
        next_state = gamma * curr + (1.0 - gamma) * torch.tanh(total_velocity)
        return self.norm(next_state).to(state.dtype)

    def circulate_mid_tensors(self, state: torch.Tensor, num_cycles: int = 1) -> torch.Tensor:
        curr = state
        for _ in range(num_cycles):
            curr = self.deliberate_step(curr)
        return curr

    def tap_efferents(self, state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        return state[:, -1, :], state[:, 0, :]


class CerebroSpheroid(nn.Module):
    def __init__(self, vocab_size: int = 32000, surface_dim: int = 2048, num_shells: int = 32, radial_dim: int = 1024, num_shortcut_tracts: int = 173):
        super().__init__()
        self.vocab_size = vocab_size
        self.surface_dim = surface_dim
        self.num_shells = num_shells
        self.sensory_dnn = SensoryInputDNN(vocab_size=vocab_size, hidden_dim=surface_dim, surface_dim=surface_dim)
        self.volumetric_core = VolumetricBrainCore(num_shells=num_shells, shell_dim=surface_dim, radial_dim=radial_dim, num_shortcut_tracts=num_shortcut_tracts)
        self.motor_dnn = MotorOutputDNN(surface_dim=surface_dim, core_dim=surface_dim, vocab_size=vocab_size)

    def ingest_stimulus(self, token_ids: torch.Tensor, state: Optional[torch.Tensor] = None) -> torch.Tensor:
        batch_size, seq_len = token_ids.shape
        if state is None:
            state = self.volumetric_core.init_state(batch_size, token_ids.device)
        for t in range(seq_len):
            flux = self.sensory_dnn(token_ids[:, t])
            state = self.volumetric_core.inject_sensory_flux(state, flux)
            state = self.volumetric_core.circulate_mid_tensors(state, num_cycles=1)
        return state

    def deliberate(self, state: torch.Tensor, thinking_cycles: int = 4) -> torch.Tensor:
        return self.volumetric_core.circulate_mid_tensors(state, num_cycles=thinking_cycles)

    def emit_next_token(self, state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        surface_efferent, core_attractor = self.volumetric_core.tap_efferents(state)
        logits = self.motor_dnn(surface_efferent, core_attractor)
        next_state = self.volumetric_core.circulate_mid_tensors(state, num_cycles=1)
        return logits, next_state

    @torch.no_grad()
    def solve_with_active_thinking(self, prompt_tokens: torch.Tensor, thinking_cycles: int = 4, max_answer_tokens: int = 25, temperature: float = 0.6) -> torch.Tensor:
        self.eval()
        state = self.ingest_stimulus(prompt_tokens)
        state = self.deliberate(state, thinking_cycles=thinking_cycles)
        generated = []
        for _ in range(max_answer_tokens):
            logits, state = self.emit_next_token(state)
            probs = F.softmax(logits / max(temperature, 1e-5), dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            generated.append(next_tok)
        return torch.cat(generated, dim=-1)

print("CerebroSpheroid cognitive architecture initialized!")

## Step 4: Instantiate Full 2.19B Model in FP16/BF16

In [ ]:
VOCAB_SIZE = 32000
print("Instantiating full 2.19B CerebroSpheroid Cognitive Model...")
model = CerebroSpheroid(
    vocab_size=VOCAB_SIZE,
    surface_dim=2048,
    num_shells=32,
    radial_dim=1024,
    num_shortcut_tracts=173,
)

total_params = sum(p.numel() for p in model.parameters())
sensory_p = sum(p.numel() for p in model.sensory_dnn.parameters())
core_p = sum(p.numel() for p in model.volumetric_core.parameters())
motor_p = sum(p.numel() for p in model.motor_dnn.parameters())

print(f"
" + "=" * 70)
print("EXACT PARAMETER BREAKDOWN (2.19B SCALE):")
print(f"  • Total Parameters            : {total_params:,} ({total_params/1e9:.3f} Billion)")
print(f"  • Afferent Sensory Input DNN  : {sensory_p:,} ({sensory_p/1e6:.2f}M params)")
print(f"  • Volumetric Spheroid Core    : {core_p:,} ({core_p/1e9:.3f} Billion params)")
print(f"  • Efferent Motor Output DNN   : {motor_p:,} ({motor_p/1e6:.2f}M params)")
print("=" * 70)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
print(f"
Loading 2.19B model onto {device} in {dtype}...")
model.to(device=device, dtype=dtype)

vram_mb = torch.cuda.memory_allocated() / (1024**2) if torch.cuda.is_available() else 0
print(f"Static VRAM: {vram_mb:.1f} MB (Comfortably fits inside Kaggle 16GB VRAM!)")

## Step 5: Connect to GSM8K Multi-Step Reasoning Dataset

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

print("Connecting to Hugging Face GSM8K reasoning dataset...")
dataset = load_dataset("openai/gsm8k", "main", split="train", streaming=True)

BATCH_SIZE = 2

def reasoning_stream():
    batch_p, batch_t = [], []
    for item in dataset:
        q = "Problem: " + item.get("question", "").strip() + " Solution:"
        a = item.get("answer", "").strip()
        if not q or not a: continue
        q_toks = [min(t, VOCAB_SIZE - 1) for t in tokenizer.encode(q)[:24]]
        a_toks = [min(t, VOCAB_SIZE - 1) for t in tokenizer.encode(a)[:16]]
        if len(q_toks) >= 6 and len(a_toks) >= 4:
            batch_p.append(q_toks)
            batch_t.append(a_toks)
            if len(batch_p) == BATCH_SIZE:
                max_p = max(len(p) for p in batch_p)
                max_t = max(len(t) for t in batch_t)
                p_pad = [p + [tokenizer.eos_token_id] * (max_p - len(p)) for p in batch_p]
                t_pad = [t + [tokenizer.eos_token_id] * (max_t - len(t)) for t in batch_t]
                yield torch.tensor(p_pad, dtype=torch.long, device=device), torch.tensor(t_pad, dtype=torch.long, device=device)
                batch_p, batch_t = [], []

stream = reasoning_stream()
print(f"Data stream initialized (Batch Size: {BATCH_SIZE})")

## Step 6: Train on Kaggle GPU with Bounded Deliberation

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

TRAIN_STEPS = 100
THINKING_CYCLES = 3
print("=" * 75)
print(f"STARTING CEREBRO-SPHEROID-2B COGNITIVE TRAINING: {TRAIN_STEPS} STEPS")
print("=" * 75)

model.train()
t_start = time.time()
losses = []

for step in range(1, TRAIN_STEPS + 1):
    prompts, targets = next(stream)
    ans_len = targets.shape[1]

    optimizer.zero_grad(set_to_none=True)
    with torch.amp.autocast(device_type="cuda" if device.type == "cuda" else "cpu", dtype=dtype):
        # Phase 1: Ingest Problem Prompts (Attractor Priming)
        with torch.no_grad():
            state = model.ingest_stimulus(prompts)
        state = state.detach().requires_grad_(True)

        # Phase 2: Active Subconscious Deliberation (Thinking without output)
        state = model.deliberate(state, thinking_cycles=THINKING_CYCLES)

        # Phase 3: Emit Answer Tokens with Zero Output Feedback!
        logits_list = []
        for t in range(ans_len):
            logits_t, state = model.emit_next_token(state)
            logits_list.append(logits_t)
        predicted_logits = torch.stack(logits_list, dim=1)
        loss = criterion(predicted_logits.view(-1, VOCAB_SIZE), targets.view(-1))

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    loss_val = loss.item()
    losses.append(loss_val)

    if step % 20 == 0 or step == 1:
        elapsed = time.time() - t_start
        peak_vram = torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0
        print(f"Step [{step:03d}/{TRAIN_STEPS:03d}] | Loss: {loss_val:.4f} | PPL: {math.exp(min(loss_val, 20)):.2f} | Peak VRAM: {peak_vram:.0f} MB | Elapsed: {elapsed:.1f}s")

total_time = time.time() - t_start
print("-" * 75)
print(f"Training Completed in {total_time:.2f}s! Final Loss: {losses[-1]:.4f}")

# Save Checkpoint in Kaggle Output Directory
save_path = "/kaggle/working/cerebro_spheroid_2b_cognitive.pt"
torch.save(model.state_dict(), save_path)
print(f"Checkpoint successfully saved to: {save_path}")

## Step 7: Demonstrate Active Thinking on New Multi-Step Problems

In [ ]:
test_queries = [
    "Problem: Maya has 8 markers. She buys 4 more and gives 3 away. How many markers does she have? Solution:",
    "Problem: A train travels 60 miles in 1 hour. How many miles does it travel in 3 hours? Solution:",
]

print("=" * 75)
print("ACTIVE DELIBERATION TEST: INGESTION -> 4 THINKING CYCLES -> ZERO-FEEDBACK ANSWER")
print("=" * 75)

model.eval()
with torch.no_grad():
    for q in test_queries:
        print(f"
[Problem]: "{q}"")
        toks = [min(t, VOCAB_SIZE - 1) for t in tokenizer.encode(q)]
        prompt_tensor = torch.tensor([toks], device=device)
        ans_toks = model.solve_with_active_thinking(prompt_tensor, thinking_cycles=4, max_answer_tokens=20, temperature=0.6)
        ans_text = tokenizer.decode(ans_toks[0].tolist(), skip_special_tokens=True)
        print(f"  [Cognitive Answer]: "{ans_text}"")
        print("  ✓ 4 internal thinking cycles executed before speech.")
        print("  ✓ Zero emitted tokens fed back into the model!")